In [1]:
from beacon_api import *
from pyproj import Transformer
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import contextily as ctx
import matplotlib.pyplot as plt
import datetime

In [2]:
exv_variable = "EXV002"                 # Surface temperature
region = (0, 20, 45, 55)                # lon_min, lon_max, lat_min, lat_max 
time = ("2000-01-01", "2025-12-31")     # date_min, date_max
height = (0, 200)                      # elevation 

In [3]:
client = Client("https://beacon-iriscc.maris.nl")
tables = client.list_tables()

Connected to: https://beacon-iriscc.maris.nl/ server successfully
Beacon Version: 1.5.4


In [ ]:
#tables['actris-in-situ'].get_table_schema()

In [4]:
query_builder = tables["actris-in-situ"].query()
query_builder.add_select_column("time", "TIME")
query_builder.add_select_column(".geospatial_lat_min", 'LATITUDE')
query_builder.add_select_column(".geospatial_lon_min", 'LONGITUDE')
query_builder.add_select_column(".ebas_station_altitude", 'HEIGHT')
query_builder.add_select_column(".ebas_framework_acronym", 'framework')
query_builder.add_select_column(".ebas_station_code", "station_code")
query_builder.add_select_column('temperature')
query_builder.add_select_column('temperature.units', 'unit')
query_builder.add_range_filter("TIME", time[0], time[1])
query_builder.add_range_filter("LONGITUDE", region[0],region[1])
query_builder.add_range_filter("LATITUDE", region[2], region[3])
query_builder.add_range_filter("HEIGHT", height[0], height[1])
query_builder.add_is_not_null_filter('temperature')
query_builder.add_range_filter('temperature', 273.15, 313.15)
df_actris = query_builder.to_pandas_dataframe()
df_actris['RI'] = 'ACTRIS'
#df_actris['EXV002'] = df_actris['EXV002'] - 273.15
df_actris

Creating JSONQuery with from: FromTable(table='actris-in-situ')
Running query: {"output": {"format": "parquet"}, "select": [{"column": "time", "alias": "TIME"}, {"column": ".geospatial_lat_min", "alias": "LATITUDE"}, {"column": ".geospatial_lon_min", "alias": "LONGITUDE"}, {"column": ".ebas_station_altitude", "alias": "HEIGHT"}, {"column": ".ebas_framework_acronym", "alias": "framework"}, {"column": ".ebas_station_code", "alias": "station_code"}, {"column": "temperature", "alias": null}, {"column": "temperature.units", "alias": "unit"}], "filters": [{"column": "TIME", "gt_eq": "2000-01-01", "lt_eq": "2025-12-31"}, {"column": "LONGITUDE", "gt_eq": 0, "lt_eq": 20}, {"column": "LATITUDE", "gt_eq": 45, "lt_eq": 55}, {"column": "HEIGHT", "gt_eq": 0, "lt_eq": 200}, {"is_not_null": {"column": "temperature"}}, {"column": "temperature", "gt_eq": 273.15, "lt_eq": 313.15}], "distinct": null, "sort_by": null, "limit": null, "offset": null, "from": "actris-in-situ"}


,TIME,LATITUDE,LONGITUDE,HEIGHT,framework,station_code,temperature,unit,RI
0,2023-02-23 17:30:00,45.478344,9.231443,118.0 m,"ATMO-ACCESS, RI-URBANS",IT0025U,293.57,K,ACTRIS
1,2023-02-23 18:30:00,45.478344,9.231443,118.0 m,"ATMO-ACCESS, RI-URBANS",IT0025U,295.52,K,ACTRIS
2,2023-02-23 19:30:00,45.478344,9.231443,118.0 m,"ATMO-ACCESS, RI-URBANS",IT0025U,296.01,K,ACTRIS
3,2023-02-23 20:30:00,45.478344,9.231443,118.0 m,"ATMO-ACCESS, RI-URBANS",IT0025U,296.13,K,ACTRIS
4,2023-02-23 21:30:00,45.478344,9.231443,118.0 m,"ATMO-ACCESS, RI-URBANS",IT0025U,296.05,K,ACTRIS
...,...,...,...,...,...,...,...,...,...
1931926,2022-09-22 16:30:00,45.772223,2.964886,1465.0 m,"ACTRIS, GAW-WDCA",FR0030R,298.90,K,ACTRIS
1931927,2022-09-22 17:30:00,45.772223,2.964886,1465.0 m,"ACTRIS, GAW-WDCA",FR0030R,298.80,K,ACTRIS
1931928,2022-09-22 18:30:00,45.772223,2.964886,1465.0 m,"ACTRIS, GAW-WDCA",FR0030R,298.50,K,ACTRIS
1931929,2022-09-22 19:30:00,45.772223,2.964886,1465.0 m,"ACTRIS, GAW-WDCA",FR0030R,298.40,K,ACTRIS


In [ ]:
import requests

def execute_sparql_query(endpoint, sparql_query):
    response = requests.get(
        endpoint,
        params={"query": sparql_query, "format": "application/sparql-results+json"},
        headers={"Accept": "application/sparql-results+json"}
    )
    
    response.raise_for_status()

    results = response.json()
    
    return results

def exv_to_p07(exv_code: str, return_preflabel: bool = False, cache: bool = True):
  
    exv_identifiers = map(lambda exv_code: f'<http://vocab.nerc.ac.uk/collection/EXV/current/{exv_code}/>', [exv_code])
    exv_identifiers = "\n".join(exv_identifiers)
    
    sparql_query = f"""
    PREFIX dce: <http://purl.org/dc/elements/1.1/>
    PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
    PREFIX iadopt: <https://w3id.org/iadopt/ont#> 
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

    SELECT DISTINCT ?p07 ?prefLabel ?notation
    WHERE {{
        VALUES ?exv {{
            {exv_identifiers}
        }} 

        OPTIONAL {{?exv iadopt:hasApplicableMatrix ?matrix .}}
        ?exv iadopt:hasApplicableObjectOfInterest ?ooi .
        ?exv iadopt:hasApplicableProperty ?property .

        <http://vocab.nerc.ac.uk/collection/P07/current/> skos:member ?p07 .

        OPTIONAL {{ ?p07 iadopt:hasMatrix ?matrix . }}
        ?p07 iadopt:hasObjectOfInterest ?ooi .
        ?p07 iadopt:hasProperty ?property .

        OPTIONAL {{ ?p07 skos:prefLabel ?prefLabel .
                FILTER(LANG(?prefLabel) = "en")
        }}
        OPTIONAL {{ ?p07 skos:notation ?notation . }}
    }}
    """

    results = execute_sparql_query('https://vocab.nerc.ac.uk/sparql', sparql_query)
    
    return_val = []
    
    if return_preflabel:
        for result in results["results"]["bindings"]:
            preflabel = result.get("prefLabel", {}).get("value", "")
            return_val.append(preflabel)
            
    else:
        # Show results
        for result in results["results"]["bindings"]:
            uri = result.get("p07", {}).get("value", "")
            return_val.append(uri.rstrip("/").split("/")[-1])
   
    return return_val

In [ ]:
#exv_to_p07("EXV002", return_preflabel = True)

In [ ]:
iagos_schema = tables['iagos-l2'].get_table_schema()
std_cols = [col for col in iagos_schema.names if col.endswith('.standard_name')]
query_builder = tables["iagos-l2"].query()

for parameter in std_cols:
    query_builder.add_select_column(parameter)

IAGOS_variable_names = query_builder.to_pandas_dataframe()

In [ ]:
value = "air_temperature"

matches = [
    col.split('.')[0]   
    for col in IAGOS_variable_names.columns
    if col.endswith('.standard_name') and IAGOS_variable_names[col].eq(value).any()
]

matches

In [ ]:
#tables["iagos-l1"].get_table_schema()

In [5]:
df_IAGOS_l1 =(
        tables["iagos-l1"]
        .query()
        .add_select_column("lon", "LONGITUDE")
        .add_select_column("lat", "LATITUDE")
        .add_select_column("UTC_time", "TIME")
        .add_select_column("baro_alt_AC", "HEIGHT")
        .add_select_column("air_temp_AC", 'air_temp_aircraft')
        .add_select_column("air_temp_AC.units", 'air_temp_aircraft_unit')
        .add_select_column("flight_name")
        .add_select_column('title')
        .add_select_column("processing_level")
        .add_range_filter("TIME", time[0], time[1])
        .add_range_filter("LONGITUDE", region[0],region[1])
        .add_range_filter("LATITUDE", region[2], region[3])
        .add_range_filter("HEIGHT", height[0], height[1])
        .add_range_filter('air_temp_aircraft', 273.15, 313.15)
        .to_pandas_dataframe()
    )

df_IAGOS_l1['RI'] = 'IAGOS'

Creating JSONQuery with from: FromTable(table='iagos-l1')
Running query: {"output": {"format": "parquet"}, "select": [{"column": "lon", "alias": "LONGITUDE"}, {"column": "lat", "alias": "LATITUDE"}, {"column": "UTC_time", "alias": "TIME"}, {"column": "baro_alt_AC", "alias": "HEIGHT"}, {"column": "air_temp_AC", "alias": "air_temp_aircraft"}, {"column": "air_temp_AC.units", "alias": "air_temp_aircraft_unit"}, {"column": "flight_name", "alias": null}, {"column": "title", "alias": null}, {"column": "processing_level", "alias": null}], "filters": [{"column": "TIME", "gt_eq": "2000-01-01", "lt_eq": "2025-12-31"}, {"column": "LONGITUDE", "gt_eq": 0, "lt_eq": 20}, {"column": "LATITUDE", "gt_eq": 45, "lt_eq": 55}, {"column": "HEIGHT", "gt_eq": 0, "lt_eq": 200}, {"column": "air_temp_aircraft", "gt_eq": 273.15, "lt_eq": 313.15}], "distinct": null, "sort_by": null, "limit": null, "offset": null, "from": "iagos-l1"}


In [6]:
df_IAGOS_l2 =(
        tables["iagos-l2"]
        .query()
        .add_select_column("lon", "LONGITUDE")
        .add_select_column("lat", "LATITUDE")
        .add_select_column("UTC_time", "TIME")
        .add_select_column("baro_alt_AC", "HEIGHT")
        .add_select_coalesced(['air_temp_PM', 'air_temp_P1'], 'air_temp_IAGOS')
        .add_select_coalesced(['air_temp_PM.units', 'air_temp_P1.units'], 'air_temp_IAGOS_units')
        .add_select_column("air_temp_AC", 'air_temp_aircraft')
        .add_select_column("air_temp_AC.units", 'air_temp_aircraft_unit')
        .add_select_column('title')
        .add_select_column("processing_level")
        .add_range_filter("TIME", time[0], time[1])
        .add_range_filter("LONGITUDE", region[0],region[1])
        .add_range_filter("LATITUDE", region[2], region[3])
        .add_range_filter("HEIGHT", height[0], height[1])
        .add_range_filter('air_temp_IAGOS', 273.15, 313.15)
        .add_range_filter('air_temp_aircraft', 273.15, 313.15)
        .to_pandas_dataframe()
    )

df_IAGOS_l2['RI'] = 'IAGOS'

Creating JSONQuery with from: FromTable(table='iagos-l2')
Running query: {"output": {"format": "parquet"}, "select": [{"column": "lon", "alias": "LONGITUDE"}, {"column": "lat", "alias": "LATITUDE"}, {"column": "UTC_time", "alias": "TIME"}, {"column": "baro_alt_AC", "alias": "HEIGHT"}, {"function": "coalesce", "args": [{"column": "air_temp_PM", "alias": null}, {"column": "air_temp_P1", "alias": null}], "alias": "air_temp_IAGOS"}, {"function": "coalesce", "args": [{"column": "air_temp_PM.units", "alias": null}, {"column": "air_temp_P1.units", "alias": null}], "alias": "air_temp_IAGOS_units"}, {"column": "air_temp_AC", "alias": "air_temp_aircraft"}, {"column": "air_temp_AC.units", "alias": "air_temp_aircraft_unit"}, {"column": "title", "alias": null}, {"column": "processing_level", "alias": null}], "filters": [{"column": "TIME", "gt_eq": "2000-01-01", "lt_eq": "2025-12-31"}, {"column": "LONGITUDE", "gt_eq": 0, "lt_eq": 20}, {"column": "LATITUDE", "gt_eq": 45, "lt_eq": 55}, {"column": "HEI

In [7]:
df_IAGOS = pd.concat([df_IAGOS_l1, df_IAGOS_l2], axis=0).reset_index(drop=True)
df_IAGOS['air_temp_IAGOS'] = df_IAGOS['air_temp_IAGOS'] - 273.15
df_IAGOS['air_temp_aircraft'] = df_IAGOS['air_temp_aircraft'] - 273.15
df_IAGOS

,LONGITUDE,LATITUDE,TIME,HEIGHT,air_temp_aircraft,air_temp_aircraft_unit,flight_name,title,processing_level,RI,air_temp_IAGOS,air_temp_IAGOS_units
0,8.5978,50.0475,2024-05-05 03:02:06.999999744,199.3,13.2,K,2024050417034704,IAGOS Intermediate Observational Data L1 - Tim...,Level 1 (Intermediate observational data): The...,IAGOS,NaN,NaN
1,8.5942,50.0467,2024-05-05 03:02:10.999999744,183.8,13.5,K,2024050417034704,IAGOS Intermediate Observational Data L1 - Tim...,Level 1 (Intermediate observational data): The...,IAGOS,NaN,NaN
2,8.5906,50.0458,2024-05-05 03:02:14.999999744,167.0,13.5,K,2024050417034704,IAGOS Intermediate Observational Data L1 - Tim...,Level 1 (Intermediate observational data): The...,IAGOS,NaN,NaN
3,8.5870,50.0450,2024-05-05 03:02:18.999999744,150.9,13.8,K,2024050417034704,IAGOS Intermediate Observational Data L1 - Tim...,Level 1 (Intermediate observational data): The...,IAGOS,NaN,NaN
4,8.5831,50.0441,2024-05-05 03:02:22.999999744,136.2,13.8,K,2024050417034704,IAGOS Intermediate Observational Data L1 - Tim...,Level 1 (Intermediate observational data): The...,IAGOS,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
159286,2.5854,48.9973,2022-10-15 11:53:29.999999744,135.9,15.8,K,NaN,IAGOS final quality controlled Observational D...,Level 2 (Final quality controlled observationa...,IAGOS,16.450006,K
159287,2.5814,48.9971,2022-10-15 11:53:33.999999744,136.2,15.8,K,NaN,IAGOS final quality controlled Observational D...,Level 2 (Final quality controlled observationa...,IAGOS,16.300012,K
159288,2.5777,48.9968,2022-10-15 11:53:37.999999744,119.2,15.8,K,NaN,IAGOS final quality controlled Observational D...,Level 2 (Final quality controlled observationa...,IAGOS,16.459985,K
159289,2.5732,48.9966,2022-10-15 11:53:41.999999744,138.4,15.5,K,NaN,IAGOS final quality controlled Observational D...,Level 2 (Final quality controlled observationa...,IAGOS,16.390009,K


In [ ]:
datasets_ICOS = client.list_datasets()
datasets_ICOS['icos/icos.bbf'].get_schema()

In [ ]:
df_ICOS = (
        datasets_ICOS['icos/icos.bbf']
        .query()
        .add_select_column("Latitude", "LATITUDE")
        .add_select_column("Longitude", "LONGITUDE")
        .add_select_column("Atmospheric Pressure [hPa]", "HEIGHT")
        .add_select_column("Atmospheric Temperature [degC]", exv_variable)
        .add_select_column("Date/Time", "TIME")
        .add_range_filter("LONGITUDE", region[0], region[1])
        .add_range_filter("LATITUDE", region[2], region[3])
        .add_range_filter("TIME", time[0], time[1])
        #.add_range_filter("HEIGHT", height[0], height[1])
        .add_is_not_null_filter(exv_variable)
        .to_pandas_dataframe()
    )

df_ICOS['RI'] = "ICOS"
df_ICOS

In [ ]:
df_merged = pd.concat([df_actris, df_IAGOS, df_ICOS], axis=0).reset_index(drop=True)
df_merged

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)

markers = ["o", "s", "^"]  # circle, square, triangle

groups = list(df_merged.groupby("RI", observed=True))
groups = sorted(groups, key=lambda x: x[0] == "ICOS")

for (ri, group), marker in zip(groups, markers):

    is_icos = ri == "ICOS"

    sc = ax.scatter(
        group["LONGITUDE"],
        group["LATITUDE"],
        c=group["EXV002"],
        cmap="viridis",
        s=30 if is_icos else 10,
        alpha=1.0 if is_icos else 0.6,
        marker=marker,
        label=ri,
        vmin=0,
        vmax=40,
        zorder=3 if is_icos else 2
    )

xmin, xmax = df_merged["LONGITUDE"].min(), df_merged["LONGITUDE"].max()
ymin, ymax = df_merged["LATITUDE"].min(), df_merged["LATITUDE"].max()
pad_x = (xmax - xmin) * 0.1
pad_y = (ymax - ymin) * 0.1

ax.set_xlim(xmin - pad_x, xmax + pad_x)
ax.set_ylim(ymin - pad_y, ymax + pad_y)

ctx.add_basemap(
    ax,
    source=ctx.providers.CartoDB.Voyager,
    crs="EPSG:4326"
)

ax.legend(title="Collection (RI)")

ax.set_title("EXV02")
ax.set_xlabel("lon")
ax.set_ylabel("lat")

cbar = fig.colorbar(sc, ax=ax, fraction=0.035, pad=0.04)
cbar.set_label("temp")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4), dpi=100)
plt.plot(df_ICOS['TIME'], df_ICOS['EXV002'], '.', label='ICOS')
plt.xlabel('Time')
plt.ylabel('Temperature [C]')
plt.legend(title='RI type')
plt.title('EXV02 timeseries by ICOS')
plt.show()

In [ ]:
plt.figure(figsize=(10, 4), dpi=100)
plt.plot(df_actris['TIME'], df_actris['EXV002'], '.', label='ACTRIS')
plt.xlabel('Time')
plt.ylabel('Temperature [C]')
plt.legend(title='RI type')
plt.title('EXV02 timeseries by ACTRIS')
plt.show()

In [ ]:
plt.figure(figsize=(10, 4), dpi=100)
plt.plot(df_IAGOS['TIME'], df_IAGOS[exv_variable], '.', label=exv_variable)
plt.xlabel('Time')
plt.ylabel('Temperature [C]')
plt.legend(title='RI type')
plt.title('EXV02 timeseries by IAGOS')
plt.show()

In [ ]:
plt.figure(figsize=(10, 4), dpi=100)
for param in ['air_temp_AC', 'air_stag_temp_PM', 'air_stag_temp_AC', 'air_temp_PM', 'air_temp_P1', 'air_stag_temp_P1']:
    plt.plot(df_IAGOS['TIME'], df_IAGOS[param], '.', label=param)


plt.xlabel('Time')
plt.ylabel('Temperature [C]')
plt.legend(title='RI type')
plt.title('EXV02 timeseries by IAGOS')
plt.show()